In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
import sqlite3
from sklearn.preprocessing import StandardScaler  # For scaling features before ML

In [2]:
# Step 2a: Main financial dataset (revenue, expense, cashflow match)
main_data = {
    'Startup Name': ['FinServe AI', 'EcoBuild Solutions', 'HealthNova Pvt Ltd', 'AgroTech India'],
    'Revenue (₹ Cr)': [120, 75, 150, 60],
    'Expense (₹ Cr)': [96, 70, 140, 55],
    'Cashflow Match %': [97, 89, 72, 95]
}
df_main = pd.DataFrame(main_data)

# Step 2b: AI Audit Scoring Breakdown (metrics as features for ML)
metrics_data = {
    'Metric': ['Transaction Consistency', 'Expense Correlation', 'Cashflow Stability', 'Debt-to-Asset Health'],
    'Weight (%)': [35, 25, 25, 15],
    'FinServe AI': [95, 90, 94, 85],
    'EcoBuild Solutions': [80, 75, 78, 74],
    'HealthNova Pvt Ltd': [50, 55, 40, 47],
    'AgroTech India': [90, 88, 89, 91]
}
df_metrics = pd.DataFrame(metrics_data)

# Extract startup columns for easy access
startups = ['FinServe AI', 'EcoBuild Solutions', 'HealthNova Pvt Ltd', 'AgroTech India']
weights = np.array(df_metrics['Weight (%)']) / 100  # Normalize weights to [0,1]

In [3]:
# Step 3: Compute weighted integrity index for each startup
integrity_scores = {}
for startup in startups:
    scores = np.array(df_metrics[startup])  # Metric scores for this startup
    weighted_score = np.dot(scores, weights)  # Dot product for weighted sum
    integrity_scores[startup] = round(weighted_score, 0)

print("Base Integrity Scores:", integrity_scores)
# Output: {'FinServe AI': 92.0, 'EcoBuild Solutions': 77.0, 'HealthNova Pvt Ltd': 48.0, 'AgroTech India': 88.0}

Base Integrity Scores: {'FinServe AI': np.float64(92.0), 'EcoBuild Solutions': np.float64(77.0), 'HealthNova Pvt Ltd': np.float64(48.0), 'AgroTech India': np.float64(89.0)}


In [5]:
# Step 4: Apply ML-Based Anomaly Detection (Isolation Forest) - FIXED LINE HERE
features = df_metrics[startups].T.values
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)
iso_forest = IsolationForest(contamination=0.25, random_state=42)
anomaly_labels = iso_forest.fit_predict(features_scaled)
anomaly_decision_scores = iso_forest.decision_function(features_scaled)

# FIXED: Add () to .values() to make it callable/iterable
normalized_penalty = (1 - (anomaly_decision_scores - anomaly_decision_scores.min()) /
                      (anomaly_decision_scores.max() - anomaly_decision_scores.min())) * 10
adjusted_integrity = np.array(list(integrity_scores.values())) - (anomaly_labels == -1) * normalized_penalty

df_main['Anomaly Score (0-100)'] = np.round(adjusted_integrity).astype(int)
for i, startup in enumerate(startups):
    integrity_scores[startup] = df_main.loc[i, 'Anomaly Score (0-100)']

print("Anomaly Labels:", dict(zip(startups, anomaly_labels)))
print("Adjusted Integrity Scores:", dict(zip(startups, df_main['Anomaly Score (0-100)'])))

Anomaly Labels: {'FinServe AI': np.int64(1), 'EcoBuild Solutions': np.int64(1), 'HealthNova Pvt Ltd': np.int64(-1), 'AgroTech India': np.int64(1)}
Adjusted Integrity Scores: {'FinServe AI': 92, 'EcoBuild Solutions': 77, 'HealthNova Pvt Ltd': 38, 'AgroTech India': 89}


In [6]:
# Step 5: Assign Ratings, Risk Levels, and Generate Remarks
def get_rating(score):
    if score >= 90: return 'Excellent'
    elif score >= 80: return 'Strong'
    elif score >= 70: return 'Moderate'
    else: return 'Poor'

def get_risk(score):
    if score >= 80: return 'Low Risk'
    elif score >= 60: return 'Moderate Risk'
    else: return 'High Risk'

df_main['Integrity Rating'] = df_main['Anomaly Score (0-100)'].apply(get_rating)
df_main['Risk Level'] = df_main['Anomaly Score (0-100)'].apply(get_risk)

def generate_remark(row, anomaly_label):
    base = row['Startup Name']
    score = row['Anomaly Score (0-100)']
    if anomaly_label == -1:
        return f"High anomaly detected in {base}: Flagged for unexplained variances; immediate audit recommended."
    elif score >= 90:
        return f"Stable records for {base}: Minimal variance in flows."
    elif score >= 80:
        return f"Consistent entries for {base}: Matches banking data."
    else:
        return f"Inconsistencies in {base}: Review vendor payments and cash deposits."

label_map = dict(zip(startups, anomaly_labels))
df_main['AI Remark'] = [generate_remark(row, label_map[row['Startup Name']]) for _, row in df_main.iterrows()]

print("\nMock Financial Integrity Dataset:")
print(df_main.to_string(index=False))


Mock Financial Integrity Dataset:
      Startup Name  Revenue (₹ Cr)  Expense (₹ Cr)  Cashflow Match %  Anomaly Score (0-100) Integrity Rating    Risk Level                                                                                                    AI Remark
       FinServe AI             120              96                97                     92        Excellent      Low Risk                                                   Stable records for FinServe AI: Minimal variance in flows.
EcoBuild Solutions              75              70                89                     77         Moderate Moderate Risk                             Inconsistencies in EcoBuild Solutions: Review vendor payments and cash deposits.
HealthNova Pvt Ltd             150             140                72                     38             Poor     High Risk High anomaly detected in HealthNova Pvt Ltd: Flagged for unexplained variances; immediate audit recommended.
    AgroTech India              60   

In [7]:
# Step 6: Generate Final Tables
print("\nAI Audit Scoring Breakdown:")
print(df_metrics.to_string(index=False))

df_final = pd.DataFrame({
    'Startup': startups,
    'Integrity Index (0-100)': df_main['Anomaly Score (0-100)'],
    'Risk Level': df_main['Risk Level']
})
print("\nFinal Financial Integrity Index:")
print(df_final.to_string(index=False))


AI Audit Scoring Breakdown:
                 Metric  Weight (%)  FinServe AI  EcoBuild Solutions  HealthNova Pvt Ltd  AgroTech India
Transaction Consistency          35           95                  80                  50              90
    Expense Correlation          25           90                  75                  55              88
     Cashflow Stability          25           94                  78                  40              89
   Debt-to-Asset Health          15           85                  74                  47              91

Final Financial Integrity Index:
           Startup  Integrity Index (0-100)    Risk Level
       FinServe AI                       92      Low Risk
EcoBuild Solutions                       77 Moderate Risk
HealthNova Pvt Ltd                       38     High Risk
    AgroTech India                       89      Low Risk


In [8]:
# Step 7: Integrate SQL for Data Persistence and Querying
conn = sqlite3.connect(':memory:')
df_main.to_sql('financial_integrity', conn, if_exists='replace', index=False)
df_metrics.to_sql('audit_metrics', conn, if_exists='replace', index=False)

high_risk_query = """
SELECT "Startup Name", "Anomaly Score (0-100)", "Risk Level", "AI Remark"
FROM financial_integrity
WHERE "Risk Level" = 'High Risk'
"""
df_high_risk = pd.read_sql_query(high_risk_query, conn)
print("\nHigh-Risk Startups (SQL Query):")
print(df_high_risk.to_string(index=False))

conn.close()


High-Risk Startups (SQL Query):
      Startup Name  Anomaly Score (0-100) Risk Level                                                                                                    AI Remark
HealthNova Pvt Ltd                     38  High Risk High anomaly detected in HealthNova Pvt Ltd: Flagged for unexplained variances; immediate audit recommended.
